# 머신러닝 적용을 위한 데이터 처리 I
## 탐색적 데이터 분석(EDA) 실습 튜토리얼 (30문제)

**학습 목표**
- EDA 4단계(데이터 확인 → 정제 → 특성 엔지니어링 → 상관관계 분석)를 순서대로 수행한다.
- Pandas 5대장(선택·정렬·카운트·조작·집계)을 `.loc[]` 중심으로 익힌다.
- Matplotlib·Seaborn으로 Line, Scatter, Bar, Hist, KDE, Jointplot, Pairplot 스타일 차트를 그린다.
- 시각화 인사이트를 `scipy.stats` 검정으로 뒷받침한다.

**필수 실습 규칙**
1. 시각화: `fig, ax = plt.subplots()` + `sns.*(..., ax=ax)`
2. Pandas: 필터·선택·조작은 `.loc[]` 명시 사용
3. 데이터: `seaborn` 내장 + Line plot은 `yfinance` 주가
4. 구조: [개념 설명] → [실습 코드] → [결과 해석]


---
## 0. 환경 설정 및 EDA 개요


### 문제 1. 라이브러리 임포트 및 시각화 환경 설정

**[개념 설명]**
EDA는 **가설 검증 전 데이터가 스스로 말하도록** 만드는 과정입니다. 분석 전 `numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`를 준비하고 한글 폰트를 설정해 차트 레이블이 깨지지 않게 합니다.


In [ ]:
# fig, ax 구조를 쓰려면 matplotlib.pyplot을 plt 별칭으로 임포트
import sys
if sys.platform != 'darwin':
    !sudo apt-get install -y -qq fonts-nanum 2>/dev/null
    !fc-cache -fv 2>/dev/null
    !rm -rf ~/.cache/matplotlib 2>/dev/null

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트: ax.set_title 등 텍스트 렌더링에 사용
for d in ['/usr/share/fonts/truetype/nanum', '/Library/Fonts']:
    for ff in fm.findSystemFonts(fontpaths=[d]):
        fm.fontManager.addfont(ff)

plt.rcParams.update({
    'font.family': 'NanumBarunGothic',
    'axes.unicode_minus': False,
    'figure.dpi': 110,
})
sns.set_theme(style='whitegrid', font='NanumBarunGothic',
              rc={'axes.unicode_minus': False})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('환경 설정 완료')


**[결과 해석]** `plt.rcParams`와 `sns.set_theme`로 이후 모든 `fig, ax` 차트에 동일한 스타일이 적용됩니다.


### 문제 2. Seaborn 내장 데이터셋 로드

**[개념 설명]**
`penguins`(수치·범주 혼합), `titanic`(분류 EDA), `tips`(회귀·팁 관계)를 로드합니다. 데이터셋마다 EDA 관점이 달라 비교 학습에 유리합니다.


In [ ]:
penguins = sns.load_dataset('penguins')
titanic = sns.load_dataset('titanic')
tips = sns.load_dataset('tips')

datasets = {'penguins': penguins, 'titanic': titanic, 'tips': tips}
for name, df in datasets.items():
    print(f'{name:10s} shape={df.shape}')


**[결과 해석]** 행·열 크기를 먼저 확인하면 메모리 부담과 변수 규모를 가늠할 수 있습니다.


### 문제 3. EDA 4단계 과정 이해

**[개념 설명]**
| 단계 | 목적 | 대표 작업 |
|------|------|----------|
| 1. 데이터 확인 | 구조·타입·분포 파악 | `head()`, `info()`, `describe()` |
| 2. 정제 | 분석 가능 형태로 정리 | 결측·중복·타입 변환 |
| 3. 특성 엔지니어링 | 변수 재구성 | 파생 변수, 구간화, 인코딩 |
| 4. 상관관계 분석 | 변수 간 관계 | `corr()`, heatmap, scatter |

이후 문제 4~30은 위 4단계 + Pandas 5대장 + 시각화 + 통계 검정 순으로 진행합니다.


In [ ]:
eda_steps = [
    '1. 데이터 확인',
    '2. 정제',
    '3. 특성 엔지니어링',
    '4. 상관관계 분석',
]
for i, step in enumerate(eda_steps, 1):
    print(f'  [{i}] {step}')


---
## 1. EDA 1단계 — 데이터 확인 (Data Inspection)


### 문제 4. `head()`, `info()`, `describe()`로 1차 스캔

**[개념 설명]**
수치·범주 변수의 **실제 값**, **결측**, **기술통계**를 한 번에 확인합니다.


In [ ]:
# penguins를 대표 데이터로 사용
print('=== head (상위 5행) ===')
display(penguins.head())

print('\n=== info (타입·결측) ===')
penguins.info()

print('\n=== describe (수치 요약) ===')
display(penguins.describe())


**[결과 해석]** `bill_length_mm` 등 수치 변수의 범위와 `species` 범주 수를 파악합니다. `info`에서 object vs float64 구분이 전처리·시각화 방법 선택의 단서가 됩니다.


### 문제 5. `.loc[]`로 특정 열·행 선택 (Pandas 5대장 — 선택)

**[개념 설명]**
`.loc[행 조건, 열]`은 **명시적** 인덱싱입니다. 조건 없이 열 subset을 고를 때도 `.loc[:, cols]` 형태를 권장합니다.


In [ ]:
# 열 선택: .loc[:, 열리스트] — 모든 행, 지정 열만
cols_num = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
pg_num = penguins.loc[:, cols_num]
print('선택 열 shape:', pg_num.shape)
pg_num.head(3)


**[결과 해석]** 수치형 4개 열만 추출해 이후 상관·히스토그램 분석의 기반 테이블을 만듭니다.


### 문제 6. `.loc[]` 조건 필터링 — Adelie 펭귄만 추출

**[개념 설명]**
불리언 마스크를 `.loc[mask, :]`에 넣으면 **조건을 코드로 문서화**할 수 있습니다.


In [ ]:
mask_adelie = penguins['species'] == 'Adelie'
adelie = penguins.loc[mask_adelie, :]
print(f'Adelie: {len(adelie)}건 / 전체 {len(penguins)}건')
adelie.loc[:, ['species', 'island', 'body_mass_g']].head()


**[결과 해석]** 종별 표본 수 차이(불균형)를 확인하면 그룹 비교 시 해석에 주의할 수 있습니다.


---
## 2. Pandas 5대장 실습 (선택 · 정렬 · 카운트 · 조작 · 집계)


### 문제 7. 정렬 — `sort_values` + `.loc` 재할당

**[개념 설명]**
정렬은 **순위·극값** 탐색에 필수입니다. 원본 보존을 위해 복사본에 적용합니다.


In [ ]:
pg_sorted = penguins.copy()
# body_mass_g 내림차순: .loc로 전체 행 유지하며 정렬된 DataFrame 생성
pg_sorted = pg_sorted.loc[pg_sorted['body_mass_g'].notnull(), :]
pg_sorted = pg_sorted.sort_values('body_mass_g', ascending=False)
pg_sorted.loc[:, ['species', 'body_mass_g']].head(5)


**[결과 해석]** 최대 체중 펭귄이 어느 종인지 빠르게 확인할 수 있습니다.


### 문제 8. 카운트 — `value_counts`와 `.loc` 그룹 크기

**[개념 설명]**
범주형 **빈도**는 bar chart의 근거가 됩니다.


In [ ]:
species_counts = penguins['species'].value_counts()
print(species_counts)

# .loc로 특정 종만 카운트
for sp in ['Adelie', 'Gentoo', 'Chinstrap']:
    n = penguins.loc[penguins['species'] == sp, :].shape[0]
    print(f'  {sp}: {n}건')


**[결과 해석]** Gentoo 표본이 상대적으로 많거나 적으면 종 간 비교 시 가중치를 고려해야 합니다.


### 문제 9. 조작 — `.loc` 파생 변수 `bill_ratio` 생성

**[개념 설명]**
특성 엔지니어링의 기초: 기존 열로 **새 의미**를 만듭니다.


In [ ]:
pg_fe = penguins.copy()
mask_valid = pg_fe['bill_length_mm'].notnull() & pg_fe['bill_depth_mm'].notnull()
# .loc[mask, 'new_col'] = ... 형태로 새 열 생성
pg_fe.loc[mask_valid, 'bill_ratio'] = (
    pg_fe.loc[mask_valid, 'bill_length_mm'] / pg_fe.loc[mask_valid, 'bill_depth_mm']
)
pg_fe.loc[mask_valid, ['bill_length_mm', 'bill_depth_mm', 'bill_ratio']].head()


**[결과 해석]** 부리 길이/깊이 비율은 종 분류에 유용한 파생 특성 후보입니다.


### 문제 10. 집계 — `groupby` + `.loc` 그룹별 평균

**[개념 설명]**
집계는 **그룹 수준** 요약 통계입니다. 시각화 전 표로 먼저 확인합니다.


In [ ]:
grp_mean = (
    penguins.dropna(subset=['body_mass_g'])
    .groupby('species', as_index=False)['body_mass_g']
    .mean()
)
print('종별 평균 체중(g):')
display(grp_mean)

# Gentoo만 .loc로 추출해 단일 값 확인
gentoo_mean = grp_mean.loc[grp_mean['species'] == 'Gentoo', 'body_mass_g'].iloc[0]
print(f'Gentoo 평균: {gentoo_mean:.0f} g')


**[결과 해석]** Gentoo가 평균 체중이 가장 크다면 KDE·boxplot에서 분리가 두드러질 것입니다.


---
## 3. EDA 2단계 — 정제 (Data Cleaning)


### 문제 11. 결측치 확인 및 `.loc`로 분석 가능 행만 유지

**[개념 설명]**
EDA 중 결측이 있는 행은 분석에서 제외하거나 대체합니다. 여기서는 **완전 케이스 분석**을 위해 행을 필터링합니다.


In [ ]:
print('정제 전 결측 행 수:')
print(penguins.isnull().sum())

mask_complete = penguins.loc[:, cols_num + ['species']].notnull().all(axis=1)
pg_clean = penguins.loc[mask_complete, :].copy()
print(f'\n정제 후: {len(pg_clean)}건 (제거 {len(penguins)-len(pg_clean)}건)')


**[결과 해석]** 결측 2~3% 제거 후에도 330건 이상이면 대부분의 시각화·상관 분석이 가능합니다.


### 문제 12. 정제 전·후 분포 비교 (Hist, subplots 1×2)

**[개념 설명]**
`fig, axes = plt.subplots(1, 2)`로 **정제 효과**를 눈으로 검증합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('body_mass_g 분포: 정제 전 vs 후', fontsize=13, fontweight='bold')

# ax=axes[i] — seaborn을 특정 축에 매핑
sns.histplot(data=penguins, x='body_mass_g', kde=True, ax=axes[0], color='coral', bins=25)
axes[0].set_title(f'정제 전 (n={penguins["body_mass_g"].notnull().sum()})')

sns.histplot(data=pg_clean, x='body_mass_g', kde=True, ax=axes[1], color='steelblue', bins=25)
axes[1].set_title(f'정제 후 (n={len(pg_clean)})')

plt.tight_layout()
plt.show()


**[결과 해석]** 분포 형태가 크게 변하지 않으면 결측 제거가 편향을 크게 만들지 않았음을 시사합니다.


---
## 4. EDA 3단계 — 특성 엔지니어링 (Feature Engineering)


### 문제 13. tips — `tip_rate` 파생 및 `.loc` 구간화

**[개념 설명]**
회귀 EDA에서 **총액 대비 팁 비율**은 고객 행동 패턴을 드러냅니다.


In [ ]:
tips_eda = tips.copy()
# .loc[:, col] 할당으로 파생 변수
tips_eda.loc[:, 'tip_rate'] = tips_eda['tip'] / tips_eda['total_bill']
tips_eda.loc[:, 'bill_bin'] = pd.cut(
    tips_eda['total_bill'],
    bins=[0, 10, 20, 30, 50, 100],
    labels=['~10', '10~20', '20~30', '30~50', '50~']
)
tips_eda.loc[:, ['total_bill', 'tip', 'tip_rate', 'bill_bin']].head()


**[결과 해석]** `tip_rate`는 `total_bill`과 독립적인 패턴을 보일 수 있어 scatter 분석 가치가 있습니다.


### 문제 14. titanic — `.loc`로 생존 라벨 한글화 열 추가

**[개념 설명]**
범주 레이블을 **분석 친화적**으로 바꾸면 차트 가독성이 올라갑니다.


In [ ]:
ti = titanic.copy()
ti.loc[ti['survived'] == 0, 'surv_label'] = '사망'
ti.loc[ti['survived'] == 1, 'surv_label'] = '생존'
ti.loc[:, ['survived', 'surv_label', 'pclass', 'sex']].head()


**[결과 해석]** 이후 bar chart에서 xticklabels 대신 의미 있는 한글 레이블을 바로 사용할 수 있습니다.


---
## 5. EDA 4단계 — 상관관계 분석 (Correlation)


### 문제 15. 피어슨 상관계수 행렬 계산

**[개념 설명]**
수치형 변수 간 **선형 관계** 강도를 -1~1로 요약합니다.


In [ ]:
corr_pg = pg_clean.loc[:, cols_num].corr(method='pearson')
print('penguins 수치 변수 상관계수:')
display(corr_pg.round(3))


**[결과 해석]** `flipper_length_mm`과 `body_mass_g` 상관이 높으면 다중공선성을 염두에 둡니다.


### 문제 16. Heatmap — `fig, ax` + `sns.heatmap(..., ax=ax)`

**[개념 설명]**
상관 행렬을 **색상**으로 표현해 패턴을 한눈에 봅니다.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
# ax=ax: heatmap을 미리 만든 도화지 위에 그림
sns.heatmap(corr_pg, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, ax=ax, linewidths=0.5)
ax.set_title('penguins 수치 변수 상관 Heatmap')
plt.tight_layout()
plt.show()


**[결과 해석]** 진한 빨강(양의 상관)·파랑(음의 상관) 셀을 scatter로 추가 검증할 후보로 삼습니다.


---
## 6. Matplotlib · Seaborn 시각화 실습


### 문제 17. Line plot — `yfinance` 주가 시계열

**[개념 설명]**
시계열 EDA에서 **추세·변동성**을 확인합니다. `ax.plot`으로 명시적 축 제어.


In [ ]:
!pip install -q yfinance
import yfinance as yf

ticker = 'AAPL'
stock = yf.download(ticker, start='2023-01-01', end='2024-01-01', progress=False)
if isinstance(stock.columns, pd.MultiIndex):
    stock.columns = stock.columns.get_level_values(0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(stock.index, stock['Close'], color='steelblue', linewidth=1.2)
ax.set_title(f'{ticker} 종가 추이 (2023)')
ax.set_xlabel('날짜')
ax.set_ylabel('종가 ($)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**[결과 해석]** 상승·하락 구간, 급등락 시점을 Line plot에서 먼저 찾고 이벤트와 대조합니다.


### 문제 18. Scatter plot — total_bill vs tip

**[개념 설명]**
두 **연속형** 변수의 관계·이상치를 점으로 확인합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=tips_eda, x='total_bill', y='tip', hue='sex',
                alpha=0.7, ax=ax)
ax.set_title('total_bill vs tip (성별 색상)')
ax.set_xlabel('총 계산서 ($)')
ax.set_ylabel('팁 ($)')
plt.tight_layout()
plt.show()


**[결과 해석]** 우상향 패턴이면 계산서가 클수록 팁도 커지는 **양의 관계**입니다.


### 문제 19. Bar plot — titanic 생존·성별 빈도

**[개념 설명]**
**범주형** 변수의 크기 비교에 적합합니다.


In [ ]:
surv_sex = (
    ti.groupby(['surv_label', 'sex'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=surv_sex, x='sex', y='count', hue='surv_label',
            ax=ax, palette='muted')
ax.set_title('성별 · 생존 빈도')
ax.set_xlabel('성별')
ax.set_ylabel('인원 수')
plt.tight_layout()
plt.show()


**[결과 해석]** 여성 생존 bar가 높으면 성별과 생존율 연관 가설을 세울 수 있습니다.


### 문제 20. Histogram — bill_length 분포

**[개념 설명]**
단일 변수 **분포 형태**(정규·치우침)를 확인합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=pg_clean, x='bill_length_mm', bins=25, ax=ax, color='steelblue')
ax.set_title('bill_length_mm 히스토그램')
ax.set_xlabel('부리 길이 (mm)')
ax.set_ylabel('빈도')
plt.tight_layout()
plt.show()


**[결과 해석]** 다봉 분포면 종 혼합 효과일 수 있어 `hue='species'` KDE로 분리 확인합니다.


### 문제 21. KDE plot — 종별 body_mass 밀도

**[개념 설명]**
KDE는 **연속 분포**를 부드러운 곡선으로 비교합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=pg_clean, x='body_mass_g', hue='species',
            fill=True, alpha=0.3, ax=ax)
ax.set_title('종별 body_mass_g KDE')
ax.set_xlabel('체중 (g)')
plt.tight_layout()
plt.show()


**[결과 해석]** KDE peak가 종마다 다르면 체중만으로 종 구분이 가능할 정도로 분리될 수 있습니다.


### 문제 22. Jointplot 스타일 — Scatter + 주변 Hist (fig + GridSpec)

**[개념 설명]**
`jointplot` 대신 `fig`와 여러 `ax`를 직접 만들어 **객체지향**으로 marginal 분포를 함께 봅니다.


In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(8, 8))
gs = GridSpec(4, 4, figure=fig)
ax_sc = fig.add_subplot(gs[1:4, 0:3])
ax_top = fig.add_subplot(gs[0, 0:3], sharex=ax_sc)
ax_right = fig.add_subplot(gs[1:4, 3], sharey=ax_sc)

sns.scatterplot(data=pg_clean, x='flipper_length_mm', y='body_mass_g',
                hue='species', ax=ax_sc, legend=True)
sns.histplot(data=pg_clean, x='flipper_length_mm', ax=ax_top, color='gray', bins=20)
sns.histplot(data=pg_clean, y='body_mass_g', ax=ax_right, color='gray', bins=20)
ax_top.tick_params(labelbottom=False)
ax_right.tick_params(labelleft=False)
ax_sc.set_xlabel('flipper_length_mm')
ax_sc.set_ylabel('body_mass_g')
fig.suptitle('Jointplot 스타일: 날개 vs 체중', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**[결과 해석]** 중앙 scatter와 주변 hist를 동시에 보면 이상치·클러스터·주변 분포를 한 화면에서 해석합니다.


### 문제 23. Pairplot 스타일 — 3×3 scatter/hist 그리드

**[개념 설명]**
다변량 EDA: 변수 쌍별 scatter, 대각선 hist를 `fig, axes` 이중 루프로 구현합니다.


In [ ]:
pair_vars = ['bill_length_mm', 'flipper_length_mm', 'body_mass_g']
n = len(pair_vars)
fig, axes = plt.subplots(n, n, figsize=(10, 10))
fig.suptitle('Pairplot 스타일 (3변수)', fontsize=13, fontweight='bold')

for i, yi in enumerate(pair_vars):
    for j, xi in enumerate(pair_vars):
        ax = axes[i, j]
        if i == j:
            sns.histplot(data=pg_clean, x=xi, ax=ax, color='steelblue', bins=15)
        else:
            sns.scatterplot(data=pg_clean, x=xi, y=yi, ax=ax,
                            hue='species', legend=False, s=15, alpha=0.6)
        if i < n - 1:
            ax.set_xlabel('')
        if j > 0:
            ax.set_ylabel('')

plt.tight_layout()
plt.show()


**[결과 해석]** 대각선 제외 scatter에서 선형·비선형·군집 패턴을 변수 쌍마다 비교합니다.


---
## 7. 통합 EDA — 데이터셋별 탐색


### 문제 24. penguins — 종·섬 교차 bar + count `.loc` 검증

**[개념 설명]**
집계표와 bar chart를 **교차 검증**합니다.


In [ ]:
island_sp = (
    pg_clean.groupby(['island', 'species'], as_index=False)
    .size()
    .rename(columns={'size': 'n'})
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=island_sp, x='island', y='n', hue='species', ax=ax)
ax.set_title('섬 · 종별 표본 수')
plt.tight_layout()
plt.show()

# .loc로 Torgersen Adelie만 확인
n_ta = pg_clean.loc[
    (pg_clean['island'] == 'Torgersen') & (pg_clean['species'] == 'Adelie'), :
].shape[0]
print(f'Torgersen Adelie: {n_ta}건')


**[결과 해석]** 특정 섬에 한 종만 있으면 섬 효과와 종 효과가 confound될 수 있습니다.


### 문제 25. titanic — 객실 등급별 생존율 bar

**[개념 설명]**
분류 타깃 `survived`와 범주 `pclass` 관계를 EDA합니다.


In [ ]:
pclass_rate = (
    ti.groupby('pclass', as_index=False)['survived']
    .mean()
    .rename(columns={'survived': 'survival_rate'})
)
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=pclass_rate, x='pclass', y='survival_rate', ax=ax, palette='Blues')
ax.set_ylim(0, 1)
ax.set_title('객실 등급별 생존율')
ax.set_xlabel('등급')
ax.set_ylabel('생존율')
for i, row in pclass_rate.iterrows():
    ax.text(row['pclass'] - 1, row['survival_rate'] + 0.02,
            f"{row['survival_rate']:.2f}", ha='center')
plt.tight_layout()
plt.show()


**[결과 해석]** 1등급 생존율이 높으면 `pclass`는 모델링 시 강력한 특성 후보입니다.


### 문제 26. tips — 요일·시간별 tip_rate boxplot

**[개념 설명]**
파생 변수 `tip_rate`로 **그룹 간 분포** 차이를 봅니다.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=tips_eda, x='day', y='tip_rate', hue='time', ax=ax)
ax.set_title('요일 · 시간대별 tip_rate')
ax.set_ylabel('팁 비율')
plt.tight_layout()
plt.show()


**[결과 해석]** 특정 요일·시간에 tip_rate median이 높으면 운영·인력 배치 인사이트로 확장 가능합니다.


---
## 8. [심화 옵션] 통계 검정으로 시각화 뒷받침


### 문제 27. t-test — Gentoo vs Adelie body_mass 평균 차이

**[개념 설명]**
KDE에서 분리가 보였다면 **두 집단 평균** 차이가 유의한지 검정합니다.


In [ ]:
g_gentoo = pg_clean.loc[pg_clean['species'] == 'Gentoo', 'body_mass_g']
g_adelie = pg_clean.loc[pg_clean['species'] == 'Adelie', 'body_mass_g']
t_stat, p_val = stats.ttest_ind(g_gentoo, g_adelie, equal_var=False)

print(f'Gentoo mean: {g_gentoo.mean():.1f} g')
print(f'Adelie mean: {g_adelie.mean():.1f} g')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_val:.4e}')
print('결론 (α=0.05):', '평균 차이 유의함' if p_val < 0.05 else '유의하지 않음')


**[결과 해석]** p-value < 0.05이면 KDE에서 본 체중 차이가 우연이 아닐 가능성이 높습니다.


### 문제 28. Pearson 상관 검정 — flipper_length vs body_mass

**[개념 설명]**
`scipy.stats.pearsonr`로 상관계수와 **p-value**를 함께 봅니다.


In [ ]:
x = pg_clean.loc[:, 'flipper_length_mm']
y = pg_clean.loc[:, 'body_mass_g']
r, p_corr = stats.pearsonr(x, y)
print(f'Pearson r = {r:.4f}')
print(f'p-value   = {p_corr:.4e}')
print('해석: r>0 이면 날개가 길수록 체중이 큰 경향, p<0.05 이면 통계적으로 유의')


**[결과 해석]** Heatmap(문제 16)의 높은 상관을 통계적으로 confirm합니다.


### 문제 29. 카이제곱 — titanic 성별·생존 독립성

**[개념 설명]**
bar chart(문제 19)에서 본 **범주 간 연관**을 검정합니다.


In [ ]:
tab = pd.crosstab(ti['sex'], ti['survived'])
chi2, p_chi, dof, expected = stats.chi2_contingency(tab)
print('교차표:')
display(tab)
print(f'chi2={chi2:.4f}, p-value={p_chi:.4e}, dof={dof}')
print('결론 (α=0.05):', '성별·생존 독립 기각(연관 있음)' if p_chi < 0.05 else '독립성 유지')


**[결과 해석]** p-value가 매우 작으면 성별과 생존은 통계적으로 연관이 있다고 볼 수 있습니다.


### 문제 30. EDA 종합 — 4단계 + 5대장 + 시각화 + 검정 체크리스트

**[개념 설명]**
프로젝트마다 아래 체크리스트로 EDA 완성도를 점검합니다.


In [ ]:
checklist = {
    '1. 데이터 확인': ['head/info/describe', '.loc 열·행 선택'],
    '2. 정제': ['결측 필터', '정제 전후 hist'],
    '3. 특성 엔지니어링': ['파생 변수', '구간화/라벨'],
    '4. 상관관계': ['corr', 'heatmap'],
    'Pandas 5대장': ['선택', '정렬', '카운트', '조작', '집계'],
    '시각화': ['Line', 'Scatter', 'Bar', 'Hist', 'KDE', 'Joint', 'Pair'],
    '통계(옵션)': ['t-test', 'pearsonr', 'chi2'],
}
for section, items in checklist.items():
    print(f'[{section}]')
    for it in items:
        print(f'  - {it}')
print('\n30문제 EDA 튜토리얼 완료')


**[결과 해석]** 체크리스트를 프로젝트 README에 붙이면 팀원·미래의 나에게 EDA 재현 경로를 남길 수 있습니다.

---
## 정리

| 영역 | 핵심 |
|------|------|
| EDA 4단계 | 확인 → 정제 → FE → 상관 |
| Pandas 5대장 | 선택·정렬·카운트·조작·집계 (`.loc` 중심) |
| 시각화 | Line, Scatter, Bar, Hist, KDE, Joint, Pair |
| 통계 | t-test, pearsonr, chi-square |

**30문제 완료** — 코드를 순서대로 실행하며 EDA 전 과정을 복습하세요.
